# 🎓 Big Data Platform & Deployment Engineering: Masterclass Notebook
> **Ecosistema:** Apache Kafka (KRaft), Apache NiFi, Apache Spark (Structured Streaming), Lakehouse (MinIO/S3), Kubernetes & ArgoCD.
> **Autor:** Data Platform & Deployment Engineer Portfolio.

---

## 🎯 Objetivo de este Cuaderno
Este notebook interactivo sirve como **cuaderno de notas técnicas, laboratorio ejecutable y guía de defensa para entrevistas**. Contiene:
1. Arquitectura end-to-end y decisiones de diseño en producción.
2. Código ejecutable para ingesta, particionado y procesamiento en streaming.
3. Simulación y solución a fallos reales (*OOM*, *Data Skew*, *Consumer Lag*, *Backpressure*).
4. Manifiestos y comandos de despliegue GitOps con **ArgoCD y Kubernetes**.
5. Las 10 preguntas trampa de entrevistas y cómo responderlas como un ingeniero Senior.

In [ ]:
# 1. Verificación del Entorno y Dependencias
import os
import subprocess
import sys

print(f"🐍 Python Version: {sys.version.split()[0]}")
print(f"📂 Current Working Directory: {os.getcwd()}")

# Comprobación de Docker y Docker Compose
try:
    docker_v = subprocess.check_output(["docker", "--version"], text=True).strip()
    compose_v = subprocess.check_output(["docker", "compose", "version"], text=True).strip()
    print(f"🐳 {docker_v}")
    print(f"📦 {compose_v}")
except Exception as e:
    print(f"⚠️ Docker check error: {e}")

---
## ⚡ Módulo 1: Apache Kafka en Modo KRaft (Zero ZooKeeper)

### ¿Por qué KRaft en lugar de ZooKeeper?
Históricamente, Kafka dependía de Apache ZooKeeper para coordinar metadatos y líderes de partición. Esto provocaba:
* Doble sobrecarga operativa y dos sistemas que mantener.
* Límites de escala en el número de particiones por clúster (~200.000 máximo).
* Recuperación lenta tras la caída de un *controller* (requería cargar todo el estado desde ZooKeeper).

**KRaft (Kafka Raft Metadata Mode)** integra el consenso directamente dentro de Kafka mediante un log interno de metadatos (`@metadata` topic).
* **Failover en milisegundos:** El estado de metadatos ya está replicado en memoria en todos los brokers.
* **Escala masiva:** Soporta millones de particiones sin degradación.
* **Despliegue unificado:** Una sola imagen Docker/Kubernetes para brokers y controllers.

### Durabilidad Bancaria (`acks=all` e Idempotencia)
Para garantizar **cero pérdida de datos (*Zero Data Loss*)**:
1. `acks='all'`: El broker no responde con éxito hasta que el mensaje se ha escrito en el líder Y en todas las réplicas en sincronía (`min.insync.replicas=2`).
2. `enable.idempotence=true`: Previene duplicados si la red falla en el ACK mediante números de secuencia internos por mensaje.
3. `user_id` como clave de particionado: Garantiza que todas las transacciones de un mismo usuario vayan a la misma partición y mantengan orden temporal estricto.

In [ ]:
# 2. Generador de Eventos de Streaming con Particionado Determinista
import random
import uuid
from datetime import datetime


def generate_sample_transaction():
    """Genera un evento de transacción financiera con detección de anomalías."""
    user_id = f"user_{random.randint(1000, 1050)}"
    amount = round(random.uniform(5.0, 950.0), 2)

    # 5% de probabilidad de generar una transacción sospechosa (> 2.500 €)
    if random.random() < 0.05:
        amount = round(random.uniform(3000.0, 8500.0), 2)

    return {
        "transaction_id": str(uuid.uuid4()),
        "user_id": user_id,
        "amount": amount,
        "currency": "EUR",
        "merchant_category": random.choice(["groceries", "electronics", "fashion", "travel", "luxury"]),
        "device": random.choice(["iOS", "Android", "Web-Chrome"]),
        "country": random.choice(["ES", "US", "UK", "FR", "DE"]),
        "timestamp": datetime.utcnow().isoformat() + "Z",
    }


# Prueba de generación de un batch de 3 eventos
sample_batch = [generate_sample_transaction() for _ in range(3)]
print("📦 Ejemplo de eventos emitidos a Kafka:")
for event in sample_batch:
    print(f"Key: {event['user_id']} -> Value: €{event['amount']} ({event['merchant_category']})")

---
## 🌊 Módulo 2: Apache NiFi (Edge Ingestion, Colas y Contrapresión)

### Arquitectura de Repositorios en NiFi
NiFi separa físicamente tres capas de almacenamiento para garantizar alta velocidad sin saturar la memoria RAM:
1. **FlowFile Repository:** Metadatos (atributos clave-valor, estado en cola, punteros a disco). Gestionado con RocksDB y Write-Ahead Logs (WAL).
2. **Content Repository:** El contenido binario real del dato. Los FlowFiles son referencias inmutables; cuando un procesador transforma un dato, no duplica el fichero si no es necesario (evita derroches de E/S).
3. **Provenance Repository:** Linaje completo (*Data Provenance*). Registra cada salto, bifurcación, filtrado o descarte de un FlowFile para auditorías forenses y cumplimiento normativo (GDPR / PCI-DSS).

### El Mecanismo de Contrapresión (*Backpressure*)
¿Qué pasa cuando la red o Kafka sufren una caída o lentitud temporal?
* Sin NiFi: El emisor satura la memoria o los discos locales, provocando caídas en cascada (*Cascading Failures*).
* Con NiFi: Cada cola entre procesadores tiene límites configurados:
  * `Back Pressure Object Threshold`: ej. 10.000 FlowFiles.
  * `Back Pressure Size Threshold`: ej. 1 GB.
* **Comportamiento automático:** En cuanto la cola alcanza el umbral, NiFi **pausa automáticamente los procesadores aguas arriba**. Cuando el consumidor aguas abajo se recupera y la cola baja del umbral, la ingesta se reactiva sin perder un solo evento.

---
## ⚡ Módulo 3: Apache Spark Structured Streaming

### Watermarking: Prevención de Fugas de Memoria en Streaming
En procesamiento de eventos con estado (*stateful stream processing*), como calcular la suma de transacciones de la última hora, Spark necesita almacenar en memoria el estado intermedio.
* **El peligro sin Watermarking:** Si un evento llega 3 días tarde, ¿Spark debe mantener el estado de hace 3 días en RAM indefinidamente? La respuesta es NO: esto provoca un `OutOfMemoryError` garantizado.
* **La solución (*Watermark*):**
  ```python
  df.withWatermark("event_time", "10 minutes")
  ```
  Le dice a Spark: *"Cualquier evento que llegue con un timestamp anterior a (Tiempo Máximo Observado - 10 minutos) se descarta como tardío ('late data') y su estado se purga de la memoria."*

### Ventanas Temporales (*Tumbling vs Sliding Windows*)
* **Tumbling Window (Ventana fija disjunta):** `window(col("event_time"), "1 minute")`. Segmentos contiguos sin solapamiento (00:00-00:01, 00:01-00:02). Ideal para facturación y métricas por minuto.
* **Sliding Window (Ventana deslizante):** `window(col("event_time"), "10 minutes", "1 minute")`. Ventana de 10 minutos recalculada cada 1 minuto. Ideal para detección de anomalías y tendencias de fraude.

In [ ]:
# 3. Estructura de Consulta en PySpark Structured Streaming (Simulación Lógica)
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# Esquema estricto para parseo de JSON en streaming
transaction_schema = StructType(
    [
        StructField("transaction_id", StringType(), False),
        StructField("user_id", StringType(), False),
        StructField("amount", DoubleType(), False),
        StructField("currency", StringType(), True),
        StructField("merchant_category", StringType(), True),
        StructField("device", StringType(), True),
        StructField("country", StringType(), True),
        StructField("timestamp", StringType(), False),
    ]
)

print("✅ Esquema PySpark cargado con éxito:")
for field in transaction_schema.fields:
    print(f"  • {field.name}: {field.dataType}")

---
## 🔥 Módulo 4: Fallos Reales de Producción ("War Stories")

### 1. El Problema del Sesgo de Datos (*Data Skew*)
* **El síntoma:** En Spark UI, una tarea de 200 tarda 45 minutos mientras que las otras 199 terminaron en 10 segundos.
* **La causa raíz:** Una clave con alta frecuencia (ej. `user_id = null` o un merchant gigante como Amazon) concentra el 90% de los datos en una sola partición de shuffle.
* **La solución de ingeniero (*Salting*):**
  Añadir un sufijo aleatorio a la clave para repartirla entre N particiones distintas durante la agregación intermedia:
  ```python
  salt_df = df.withColumn("salted_key", concat(col("user_id"), lit("_"), floor(rand() * 10)))
  ```
  Y luego realizar una segunda agregación global agrupando por la clave original.

### 2. Contenedor Muerto por OOM (`Container killed by YARN/K8s`)
* **El error:** `Exit code 137` o `exceeded memory limits`.
* **La causa raíz:** Los procesos de Python (PySpark/Arrow) o los buffers de red de Netty usan memoria externa al heap de la JVM (*Off-Heap Memory*). Si la suma de Heap + Off-Heap supera el límite del contenedor en Kubernetes, el cgroup del kernel mata el pod.
* **La solución:** Aumentar el overhead de memoria en el submit:
  ```properties
  spark.executor.memoryOverhead=1024m  # Subir del 10% por defecto al 20-25%
  ```

### 3. El Problema de los Archivos Pequeños (*Small Files Problem*)
* **El síntoma:** El streaming continuo escribiendo en S3/MinIO genera millones de ficheros de pocos KB. Las consultas en Trino/Athena tardan minutos solo listando metadatos.
* **La solución:** Adoptar tablas **Apache Iceberg / Delta Lake** y programar compactaciones periódicas (`rewrite_data_files`), consolidando ficheros pequeños en bloques óptimos de 128 MB a 512 MB.

---
## 🚀 Módulo 5: Despliegue de Plataforma & GitOps (Kubernetes + ArgoCD)

### ¿Cómo se despliega una Plataforma de Datos Moderna?
En empresas de primer nivel, **nadie ejecuta `spark-submit` a mano en un servidor**. Todo se gestiona mediante infraestructura declarativa y GitOps:

1. **Strimzi Kafka Operator en K8s:**
   Gestiona Kafka mediante Custom Resource Definitions (CRDs):
   * `Kafka`: Define el clúster de brokers, almacenamiento persistente (PVCs) y réplicas.
   * `KafkaTopic`: Un fichero YAML en Git crea o redimensiona automáticamente el tópico `financial_transactions`.
2. **Spark-on-K8s (Spark Operator):**
   Los jobs de PySpark se definen en manifiestos `SparkApplication`. Kubernetes crea dinámicamente el Pod Driver y los Pods Ejecutores, y los destruye al finalizar para ahorrar costes.
3. **ArgoCD (El Controlador GitOps):**
   * Vigila la rama `main` de tu repositorio de GitHub.
   * Si actualizas el número de ejecutores de Spark o el esquema de Kafka en Git, **ArgoCD sincroniza el clúster automáticamente**.
   * Si alguien modifica algo manualmente en el clúster, ArgoCD detecta el *drift* y restaura el estado declarado en Git. Cero cambios no auditados en producción.

---
## 🎯 Módulo 6: Manual de Defensa en Entrevistas Técnicas

A continuación tienes las preguntas que diferencian a un aspirante junior de un **ingeniero contratado**:

### P1: *"¿Por qué usar Spark Structured Streaming en lugar de micro-servicios o Python puro leyendo de Kafka?"*
> **Tu respuesta:**  
> *"Spark Structured Streaming ofrece procesamiento distribuido tolerante a fallos con semántica de entrega 'exactly-once' nativa mediante checkpointing y Write-Ahead Logs. Además, permite aplicar operadores relacionales y SQL complejos con optimización Catalyst (pushdown de predicados) sobre petabytes de datos en ventanas de tiempo con watermarking, algo inviable de mantener artesanalmente en microservicios."*

### P2: *"¿Qué pasa en un consumidor de Kafka cuando el tiempo de procesamiento de un batch supera `max.poll.interval.ms`?"*
> **Tu respuesta:**  
> *"El coordinator del broker asume que el consumidor ha muerto y expulsa al consumidor del grupo, desatando un **Rebalance**. Durante el rebalance, el procesamiento se pausa para reasignar particiones. Si el siguiente consumidor también tarda demasiado con ese batch pesado, se produce un 'cascading rebalance' continuo. Se resuelve disminuyendo `max.poll.records` o desacoplando la lógica de procesamiento en un pool de hilos."*

### P3: *"¿Cómo garantizas que el lago de datos no se degrade con el tiempo al escribir streaming continuo?"*
> **Tu respuesta:**  
> *"Implementando tablas abiertas transaccionales como **Apache Iceberg o Delta Lake** sobre S3/MinIO. Estas capas admiten lecturas concurrentes sin lecturas sucias (*dirty reads*), aislamiento de instantáneas (snapshot isolation), y permiten programar compactaciones asíncronas (`rewrite_data_files`) que fusionan ficheros pequeños en fragmentos óptimos de 256 MB sin detener los pipelines de ingesta."*
